In [1]:
from pathlib import Path
import subprocess, time, os

SRC = Path("/content/drive/MyDrive/GEE_Exports")
LOCAL = Path("/content/gee_exports_local")   # uma pasta só, como você pediu

MOSAIC_DST = LOCAL / "mosaic"
GT_DST     = LOCAL / "gt"

for p in [LOCAL, MOSAIC_DST, GT_DST]:
    p.mkdir(parents=True, exist_ok=True)

print("SRC:", SRC)
print("LOCAL:", LOCAL)
print("MOSAIC_DST:", MOSAIC_DST)
print("GT_DST:", GT_DST)


SRC: /content/drive/MyDrive/GEE_Exports
LOCAL: /content/gee_exports_local
MOSAIC_DST: /content/gee_exports_local/mosaic
GT_DST: /content/gee_exports_local/gt


In [2]:
from google.colab import drive

def ensure_drive():
    # tenta acessar SRC; se falhar, remonta
    try:
        if SRC.exists():
            _ = next(SRC.iterdir())
            return
    except Exception:
        pass

    print("Remounting Drive...")
    drive.mount("/content/drive", force_remount=True)
    time.sleep(2)

    if not SRC.exists():
        raise RuntimeError("SRC não existe após mount. Verifique: /content/drive/MyDrive/GEE_Exports")

ensure_drive()
print("Drive OK ✅")


Drive OK ✅


In [3]:
import re

ensure_drive()

pat_mosaic = re.compile(r"^unet_mt_2023_mosaic_.*\.tif$", re.IGNORECASE)
pat_gt     = re.compile(r"^unet_mt_2023_gt_.*\.tif$", re.IGNORECASE)

mosaics = sorted([p for p in SRC.glob("*.tif") if pat_mosaic.match(p.name)])
gts     = sorted([p for p in SRC.glob("*.tif") if pat_gt.match(p.name)])

print("mosaic:", len(mosaics))
print("gt    :", len(gts))

assert len(mosaics) == 169, "Esperado 169 mosaics no Drive."
assert len(gts) == 169, "Esperado 169 gts no Drive."


mosaic: 169
gt    : 169


In [4]:
def cp_one(src: Path, dst: Path, max_tries=4) -> bool:
    # pula se já existe e tem tamanho > 0
    if dst.exists() and dst.stat().st_size > 1024:
        return True

    for t in range(1, max_tries + 1):
        ensure_drive()
        r = subprocess.run(
            ["bash", "-lc", f"cp -f '{src}' '{dst}'"],
            capture_output=True, text=True
        )
        if r.returncode == 0 and dst.exists() and dst.stat().st_size > 1024:
            return True

        err = (r.stderr or "") + (r.stdout or "")
        if "Transport endpoint is not connected" in err:
            print("Drive caiu. Remount e retry...", src.name)
            # força remount no próximo loop
            subprocess.run(["bash","-lc","umount /content/drive 2>/dev/null || true"], check=False)
            time.sleep(2)
            continue

        # erro qualquer (mostra e tenta de novo)
        print(f"[try {t}/{max_tries}] falhou:", src.name)
        if err.strip():
            print(err[:4000])
        time.sleep(1)

    return False


In [5]:
ok = 0
bad = []

for i, src in enumerate(mosaics, 1):
    dst = MOSAIC_DST / src.name
    if cp_one(src, dst):
        ok += 1
    else:
        bad.append(src.name)

    if i % 10 == 0:
        print(f"mosaic progress: {i}/{len(mosaics)}  ok={ok}  bad={len(bad)}")

print("mosaic done. ok:", ok, "bad:", len(bad))
if bad:
    print("bad examples:", bad[:10])


Drive caiu. Remount e retry... unet_mt_2023_mosaic_x-50p00_y-16p00.tif
mosaic progress: 10/169  ok=10  bad=0
mosaic progress: 20/169  ok=20  bad=0
mosaic progress: 30/169  ok=30  bad=0
Drive caiu. Remount e retry... unet_mt_2023_mosaic_x-52p00_y-16p00.tif
mosaic progress: 40/169  ok=40  bad=0
mosaic progress: 50/169  ok=50  bad=0
mosaic progress: 60/169  ok=60  bad=0
mosaic progress: 70/169  ok=70  bad=0
mosaic progress: 80/169  ok=80  bad=0
mosaic progress: 90/169  ok=90  bad=0
mosaic progress: 100/169  ok=100  bad=0
mosaic progress: 110/169  ok=110  bad=0
mosaic progress: 120/169  ok=120  bad=0
mosaic progress: 130/169  ok=130  bad=0
mosaic progress: 140/169  ok=140  bad=0
mosaic progress: 150/169  ok=150  bad=0
mosaic progress: 160/169  ok=160  bad=0
mosaic done. ok: 169 bad: 0


In [6]:
ok = 0
bad = []

for i, src in enumerate(gts, 1):
    dst = GT_DST / src.name
    if cp_one(src, dst):
        ok += 1
    else:
        bad.append(src.name)

    if i % 10 == 0:
        print(f"gt progress: {i}/{len(gts)}  ok={ok}  bad={len(bad)}")

print("gt done. ok:", ok, "bad:", len(bad))
if bad:
    print("bad examples:", bad[:10])


gt progress: 10/169  ok=10  bad=0
gt progress: 20/169  ok=20  bad=0
gt progress: 30/169  ok=30  bad=0
gt progress: 40/169  ok=40  bad=0
gt progress: 50/169  ok=50  bad=0
gt progress: 60/169  ok=60  bad=0
gt progress: 70/169  ok=70  bad=0
gt progress: 80/169  ok=80  bad=0
gt progress: 90/169  ok=90  bad=0
gt progress: 100/169  ok=100  bad=0
gt progress: 110/169  ok=110  bad=0
gt progress: 120/169  ok=120  bad=0
gt progress: 130/169  ok=130  bad=0
gt progress: 140/169  ok=140  bad=0
gt progress: 150/169  ok=150  bad=0
gt progress: 160/169  ok=160  bad=0
gt done. ok: 169 bad: 0


In [7]:
m_local = list(MOSAIC_DST.glob("*.tif"))
g_local = list(GT_DST.glob("*.tif"))

print("local mosaic:", len(m_local))
print("local gt    :", len(g_local))

assert len(m_local) == 169, "Faltou mosaic no local."
assert len(g_local) == 169, "Faltou GT no local."
print("OK ✅ 169/169 mosaic e 169/169 gt no local")


local mosaic: 169
local gt    : 169
OK ✅ 169/169 mosaic e 169/169 gt no local


In [8]:
import rasterio
from random import choice

m = choice(m_local)
g = choice(g_local)

with rasterio.open(m) as ds:
    print("MOSAIC:", m.name, "bands:", ds.count, "crs:", ds.crs, "shape:", (ds.height, ds.width), "dtype:", ds.dtypes[0])

with rasterio.open(g) as ds:
    print("GT:", g.name, "bands:", ds.count, "crs:", ds.crs, "shape:", (ds.height, ds.width), "dtype:", ds.dtypes[0])

MOSAIC: unet_mt_2023_mosaic_x-57p00_y-16p00.tif bands: 21 crs: EPSG:4326 shape: (3712, 3712) dtype: uint16
GT: unet_mt_2023_gt_x-61p00_y-15p00.tif bands: 3 crs: EPSG:4326 shape: (3711, 3712) dtype: uint8


In [9]:
from pathlib import Path
import re
import pandas as pd

MOSAIC_DIR = Path("/content/gee_exports_local/mosaic")
GT_DIR     = Path("/content/gee_exports_local/gt")

pat_xy = re.compile(r"_x(?P<x>-?\d+p\d+)_y(?P<y>-?\d+p\d+)\.tif$", re.IGNORECASE)

def xy_key(name: str):
    m = pat_xy.search(name)
    return None if not m else (m.group("x"), m.group("y"))

mosaics = sorted(MOSAIC_DIR.glob("*.tif"))
gts     = sorted(GT_DIR.glob("*.tif"))

m_idx = {xy_key(p.name): p for p in mosaics if xy_key(p.name)}
g_idx = {xy_key(p.name): p for p in gts     if xy_key(p.name)}

keys_all = sorted(set(m_idx) | set(g_idx))
rows = []
for k in keys_all:
    rows.append({
        "key": f"x{k[0]}_y{k[1]}" if k else "NA",
        "has_mosaic": k in m_idx,
        "has_gt": k in g_idx,
        "mosaic": m_idx.get(k).name if k in m_idx else None,
        "gt": g_idx.get(k).name if k in g_idx else None,
    })

df_pairs = pd.DataFrame(rows)
print("pairs:", len(df_pairs))
print("missing mosaics:", int((~df_pairs["has_mosaic"]).sum()))
print("missing gts    :", int((~df_pairs["has_gt"]).sum()))
df_pairs.head(10)

pairs: 169
missing mosaics: 0
missing gts    : 0


,key,has_mosaic,has_gt,mosaic,gt
0,x-50p00_y-10p00,True,True,unet_mt_2023_mosaic_x-50p00_y-10p00.tif,unet_mt_2023_gt_x-50p00_y-10p00.tif
1,x-50p00_y-11p00,True,True,unet_mt_2023_mosaic_x-50p00_y-11p00.tif,unet_mt_2023_gt_x-50p00_y-11p00.tif
2,x-50p00_y-12p00,True,True,unet_mt_2023_mosaic_x-50p00_y-12p00.tif,unet_mt_2023_gt_x-50p00_y-12p00.tif
3,x-50p00_y-13p00,True,True,unet_mt_2023_mosaic_x-50p00_y-13p00.tif,unet_mt_2023_gt_x-50p00_y-13p00.tif
4,x-50p00_y-14p00,True,True,unet_mt_2023_mosaic_x-50p00_y-14p00.tif,unet_mt_2023_gt_x-50p00_y-14p00.tif
5,x-50p00_y-15p00,True,True,unet_mt_2023_mosaic_x-50p00_y-15p00.tif,unet_mt_2023_gt_x-50p00_y-15p00.tif
6,x-50p00_y-16p00,True,True,unet_mt_2023_mosaic_x-50p00_y-16p00.tif,unet_mt_2023_gt_x-50p00_y-16p00.tif
7,x-50p00_y-17p00,True,True,unet_mt_2023_mosaic_x-50p00_y-17p00.tif,unet_mt_2023_gt_x-50p00_y-17p00.tif
8,x-50p00_y-18p00,True,True,unet_mt_2023_mosaic_x-50p00_y-18p00.tif,unet_mt_2023_gt_x-50p00_y-18p00.tif
9,x-50p00_y-19p00,True,True,unet_mt_2023_mosaic_x-50p00_y-19p00.tif,unet_mt_2023_gt_x-50p00_y-19p00.tif


In [10]:
import rasterio
import numpy as np

def meta(path: Path):
    with rasterio.open(path) as ds:
        return {
            "crs": str(ds.crs),
            "width": ds.width,
            "height": ds.height,
            "count": ds.count,
            "dtype": ds.dtypes[0],
            "transform": tuple(ds.transform)  # affine -> tupla comparável
        }

# pega só pares completos
pairs_ok = df_pairs[df_pairs["has_mosaic"] & df_pairs["has_gt"]].copy()
assert len(pairs_ok) == 169, "Esperado 169 pares completos."

# amostra 5
sample = pairs_ok.sample(5, random_state=42)
for _, r in sample.iterrows():
    m = MOSAIC_DIR / r["mosaic"]
    g = GT_DIR / r["gt"]
    mm = meta(m); gg = meta(g)
    print("\n", r["key"])
    print(" mosaic:", mm["crs"], mm["width"], mm["height"], "bands", mm["count"])
    print(" gt    :", gg["crs"], gg["width"], gg["height"], "bands", gg["count"])
    print(" same_crs:", mm["crs"] == gg["crs"])
    print(" same_shape:", (mm["width"],mm["height"]) == (gg["width"],gg["height"]))
    print(" same_transform:", mm["transform"] == gg["transform"])

# scan completo (metadados) – sem ler pixels
bad = []
for _, r in pairs_ok.iterrows():
    m = MOSAIC_DIR / r["mosaic"]
    g = GT_DIR / r["gt"]
    mm = meta(m); gg = meta(g)

    ok = (
        mm["crs"] == gg["crs"] and
        (mm["width"], mm["height"]) == (gg["width"], gg["height"]) and
        mm["transform"] == gg["transform"] and
        mm["count"] == 21 and
        gg["count"] == 3
    )
    if not ok:
        bad.append((r["key"], mm, gg))

print("bad tiles:", len(bad))
if bad:
    print("example:", bad[0][0])


 x-60p00_y-18p00
 mosaic: EPSG:4326 3711 3711 bands 21
 gt    : EPSG:4326 3711 3711 bands 3
 same_crs: True
 same_shape: True
 same_transform: True

 x-52p00_y-14p00
 mosaic: EPSG:4326 3711 3712 bands 21
 gt    : EPSG:4326 3711 3712 bands 3
 same_crs: True
 same_shape: True
 same_transform: True

 x-59p00_y-12p00
 mosaic: EPSG:4326 3712 3711 bands 21
 gt    : EPSG:4326 3712 3711 bands 3
 same_crs: True
 same_shape: True
 same_transform: True

 x-52p00_y-13p00
 mosaic: EPSG:4326 3711 3712 bands 21
 gt    : EPSG:4326 3711 3712 bands 3
 same_crs: True
 same_shape: True
 same_transform: True

 x-61p00_y-10p00
 mosaic: EPSG:4326 3712 3712 bands 21
 gt    : EPSG:4326 3712 3712 bands 3
 same_crs: True
 same_shape: True
 same_transform: True
bad tiles: 0


In [11]:
if len(bad) == 0:
    print("ALINHAMENTO OK ✅ (CRS/shape/transform iguais em todos os 169 pares)")
else:
    rows = []
    for key, mm, gg in bad:
        rows.append({
            "key": key,
            "m_crs": mm["crs"], "g_crs": gg["crs"],
            "m_shape": (mm["height"], mm["width"]),
            "g_shape": (gg["height"], gg["width"]),
            "same_transform": mm["transform"] == gg["transform"],
            "m_bands": mm["count"], "g_bands": gg["count"],
        })
    pd.DataFrame(rows).head(20)


ALINHAMENTO OK ✅ (CRS/shape/transform iguais em todos os 169 pares)


In [12]:
import numpy as np

pairs_ok = df_pairs[df_pairs["has_mosaic"] & df_pairs["has_gt"]].copy()
pairs_ok["mosaic_path"] = pairs_ok["mosaic"].apply(lambda n: str(MOSAIC_DIR / n))
pairs_ok["gt_path"]     = pairs_ok["gt"].apply(lambda n: str(GT_DIR / n))

# split por TILE (evita vazamento espacial no patching)
rng = np.random.default_rng(42)
idx = rng.permutation(len(pairs_ok))

n = len(pairs_ok)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

split = np.array(["test"] * n)
split[idx[:n_train]] = "train"
split[idx[n_train:n_train+n_val]] = "val"

pairs_ok["split_tile"] = split

print(pairs_ok["split_tile"].value_counts())

manifest_path = Path("/content/unet_tiles_manifest.csv")
pairs_ok[["key","split_tile","mosaic_path","gt_path"]].to_csv(manifest_path, index=False)
print("saved:", manifest_path)

split_tile
trai    118
test     26
val      25
Name: count, dtype: int64
saved: /content/unet_tiles_manifest.csv


In [13]:
import pandas as pd
import numpy as np
import rasterio

manifest = pd.read_csv("/content/unet_tiles_manifest.csv")
print(manifest["split_tile"].value_counts())
print("tiles:", len(manifest))

# checa 1 tile aleatório
row = manifest.sample(1, random_state=42).iloc[0]
m_path = row["mosaic_path"]
g_path = row["gt_path"]
print("sample mosaic:", m_path)
print("sample gt    :", g_path)

with rasterio.open(m_path) as ds:
    print("mosaic bands:", ds.count, "dtype:", ds.dtypes[0], "shape:", (ds.height, ds.width))
    # lê 1 banda só para estatística rápida
    b1 = ds.read(1, out_dtype="float32")
    print("band1 min/max:", float(np.nanmin(b1)), float(np.nanmax(b1)))

with rasterio.open(g_path) as ds:
    print("gt bands:", ds.count, "dtype:", ds.dtypes[0], "shape:", (ds.height, ds.width))
    gt = ds.read()  # (3,H,W)
    uniq = {i: np.unique(gt[i]).tolist()[:20] for i in range(gt.shape[0])}
    print("unique (per band) (first 20):", uniq)


split_tile
trai    118
test     26
val      25
Name: count, dtype: int64
tiles: 169
sample mosaic: /content/gee_exports_local/mosaic/unet_mt_2023_mosaic_x-60p00_y-18p00.tif
sample gt    : /content/gee_exports_local/gt/unet_mt_2023_gt_x-60p00_y-18p00.tif
mosaic bands: 21 dtype: uint16 shape: (3711, 3711)
band1 min/max: 0.0 0.0
gt bands: 3 dtype: uint8 shape: (3711, 3711)
unique (per band) (first 20): {0: [0], 1: [0], 2: [0]}


In [14]:
from pathlib import Path

PATCH = 512      # 512 ou 256
STRIDE = 512     # =PATCH (sem overlap). para overlap use 256 com PATCH=512
SCALE_U16 = 10000.0

OUT = Path("/content/unet_patches")
OUT_X = OUT / "x"   # npy float32 (H,W,C)
OUT_Y = OUT / "y"   # npy uint8   (H,W,3)
OUT.mkdir(parents=True, exist_ok=True)
OUT_X.mkdir(parents=True, exist_ok=True)
OUT_Y.mkdir(parents=True, exist_ok=True)

print("OUT:", OUT)
print("PATCH:", PATCH, "STRIDE:", STRIDE)


OUT: /content/unet_patches
PATCH: 512 STRIDE: 512


In [15]:
import math
import os
from tqdm import tqdm

def sliding_windows(height, width, patch, stride):
    for y in range(0, height - patch + 1, stride):
        for x in range(0, width - patch + 1, stride):
            yield x, y

def save_patch(x_arr, y_arr, key, split, x0, y0, idx):
    # x_arr: (C,H,W) float32 -> salva (H,W,C)
    # y_arr: (3,H,W) uint8 -> salva (H,W,3)
    x_out = np.moveaxis(x_arr, 0, -1).astype(np.float32)
    y_out = np.moveaxis(y_arr, 0, -1).astype(np.uint8)

    stem = f"{split}__{key}__x{x0}_y{y0}__{idx:05d}"
    np.save(OUT_X / f"{stem}.npy", x_out)
    np.save(OUT_Y / f"{stem}.npy", y_out)

def extract_tile_patches(mosaic_path, gt_path, key, split, patch=512, stride=512, scale_u16=10000.0,
                         min_labeled_pixels=50):
    with rasterio.open(mosaic_path) as xm, rasterio.open(gt_path) as yg:
        assert xm.width == yg.width and xm.height == yg.height
        H, W = xm.height, xm.width

        idx = 0
        kept = 0
        for x0, y0 in sliding_windows(H, W, patch, stride):
            # lê janela
            x = xm.read(window=((y0, y0+patch), (x0, x0+patch))).astype(np.float32) / scale_u16
            y = yg.read(window=((y0, y0+patch), (x0, x0+patch))).astype(np.uint8)

            # filtro: precisa ter algum rótulo válido em alguma banda (train/test/val)
            valid = (y != 255)
            n_valid = int(valid.sum())
            if n_valid < min_labeled_pixels:
                idx += 1
                continue

            save_patch(x, y, key, split, x0, y0, idx)
            kept += 1
            idx += 1

        return kept

In [16]:
print(manifest["split_tile"].value_counts(dropna=False).head(20))
print("unique:", sorted(manifest["split_tile"].dropna().unique().tolist()))


split_tile
trai    118
test     26
val      25
Name: count, dtype: int64
unique: ['test', 'trai', 'val']


In [17]:
# célula: amostragem controlada de patches (shards .npz) + limpeza opcional
from pathlib import Path
import shutil
import math
import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm import tqdm

# =========================
# 0) limpeza opcional
# =========================
CLEAN_OLD = True
OLD_BASE = Path("/content/unet_patches")  # onde você disse que já criou lixo (x/y)
if CLEAN_OLD and OLD_BASE.exists():
    shutil.rmtree(OLD_BASE, ignore_errors=True)
    print(f"removed: {OLD_BASE}")

# pasta única (uma só, porra) para saída
OUT_ROOT = Path("/content/unet_patches_v4")  # ajuste se quiser outro nome
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# 1) utilitários
# =========================
def normalize_split(s: str) -> str:
    s = str(s).strip().lower()
    if s.startswith("train") or s == "trai":
        return "train"
    if s.startswith("val"):
        return "val"
    if s.startswith("test"):
        return "test"
    return "other"

if "split_norm" not in manifest.columns:
    manifest["split_norm"] = manifest["split_tile"].map(normalize_split)

print("split_norm counts:\n", manifest["split_norm"].value_counts(dropna=False))

def disk_free_gb(path="/content") -> float:
    du = shutil.disk_usage(path)
    return du.free / (1024**3)

# integral image helpers (somatório rápido por janela)
def integral_image(a: np.ndarray) -> np.ndarray:
    return a.cumsum(axis=0).cumsum(axis=1)

def rect_sum(ii: np.ndarray, r0: int, c0: int, h: int, w: int) -> int:
    r1 = r0 + h - 1
    c1 = c0 + w - 1
    A = ii[r1, c1]
    B = ii[r0-1, c1] if r0 > 0 else 0
    C = ii[r1, c0-1] if c0 > 0 else 0
    D = ii[r0-1, c0-1] if (r0 > 0 and c0 > 0) else 0
    return int(A - B - C + D)

# =========================
# 2) parâmetros (ajuste aqui)
# =========================
SEED = 42
rng = np.random.default_rng(SEED)

PATCH = 256              # 256 é padrão; se apertar disco, use 192
BANDS = 21               # mosaico
SCALE_U16 = 10000        # mosaico está em uint16 escalado
MIN_LABELED = 200        # mínimo de pixels gt != 255
MIN_POS_PIXELS = 200     # para patch "positivo": mínimo de pixels == 1
POS_FRAC = {"train": 0.50, "val": 0.50, "test": 0.50}  # fração de patches positivos
RESERVE_GB = 1.5         # reserva de disco p/ não quebrar sessão
SHARD_SIZE = 32          # patches por shard .npz (32/64). 32 = mais seguro no disco

# estima custo por patch: X uint16 + Y uint8
bytes_per_patch = PATCH * PATCH * BANDS * 2 + PATCH * PATCH * 1
mb_per_patch = bytes_per_patch / (1024**2)
print(f"approx MB/patch @ {PATCH}: {mb_per_patch:.2f} MB")

free_gb = disk_free_gb()
max_patches = max(0, int(((free_gb - RESERVE_GB) * (1024**3)) / bytes_per_patch))
print(f"free_gb={free_gb:.2f} -> max_patches~{max_patches} (com reserve {RESERVE_GB} GB)")

# alvo total: define “orçamento” conservador (autoajusta se necessário)
TARGET = {"train": 3000, "val": 600, "test": 600}  # ajuste se quiser mais/menos
target_sum = sum(TARGET.values())
if target_sum > max_patches and max_patches > 0:
    scale = max_patches / target_sum
    TARGET = {k: max(50, int(v * scale)) for k, v in TARGET.items()}
    print("TARGET autoajustado para caber no disco:", TARGET)

print("TARGET final:", TARGET)

# =========================
# 3) amostragem por tile
# =========================
def sample_from_one_tile(mosaic_path: str, gt_path: str, n_need: int, want_pos: bool):
    """
    retorna listas X,Y (até n_need) amostrados de um tile
    X: (n, H, W, C) uint16
    Y: (n, H, W) uint8 (0/1/255)
    """
    Xs, Ys = [], []

    with rasterio.open(gt_path) as gds:
        gt = gds.read(1)  # uint8
        H, W = gt.shape

    # janela possível
    H0 = H - PATCH + 1
    W0 = W - PATCH + 1
    if H0 <= 0 or W0 <= 0:
        return Xs, Ys

    labeled = (gt != 255).astype(np.uint8)
    pos = (gt == 1).astype(np.uint8)

    ii_lab = integral_image(labeled)
    ii_pos = integral_image(pos)

    # tentativas: sobe se estiver difícil achar patches válidos
    max_tries = max(2000, n_need * 200)

    # abre mosaico só quando for extrair patch aceito
    mds = rasterio.open(mosaic_path)

    try:
        tries = 0
        while len(Xs) < n_need and tries < max_tries:
            tries += 1
            r0 = int(rng.integers(0, H0))
            c0 = int(rng.integers(0, W0))

            lab_cnt = rect_sum(ii_lab, r0, c0, PATCH, PATCH)
            if lab_cnt < MIN_LABELED:
                continue

            pos_cnt = rect_sum(ii_pos, r0, c0, PATCH, PATCH)

            if want_pos:
                if pos_cnt < MIN_POS_PIXELS:
                    continue
            else:
                # negativo: pode exigir "quase sem milho"
                if pos_cnt > 0:
                    continue

            w = Window(c0, r0, PATCH, PATCH)
            X = mds.read(window=w)          # (C, H, W)
            Y = gt[r0:r0+PATCH, c0:c0+PATCH]

            # sanity: se mosaico vier com outra shape (tile vazio), pula
            if X.shape[1] != PATCH or X.shape[2] != PATCH:
                continue

            # channels-last p/ DL (H,W,C)
            X = np.transpose(X, (1, 2, 0)).astype(np.uint16)
            Y = Y.astype(np.uint8)

            Xs.append(X)
            Ys.append(Y)

    finally:
        mds.close()

    return Xs, Ys

# =========================
# 4) loop principal + shards
# =========================
splits = ["train", "val", "test"]
stats = {s: {"kept": 0, "pos": 0, "neg": 0, "shards": 0} for s in splits}

for split in splits:
    df = manifest[manifest["split_norm"] == split].copy()
    if len(df) == 0:
        print(f"[{split}] sem tiles no manifest.")
        continue

    # embaralha tiles pra não ficar preso em uma região
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    need_total = TARGET[split]
    need_pos = int(math.ceil(need_total * POS_FRAC.get(split, 0.5)))
    need_neg = need_total - need_pos

    shard_idx = 0
    tile_idx = 0

    print(f"\n[{split}] target={need_total} (pos={need_pos} neg={need_neg}) tiles={len(df)} free_gb={disk_free_gb():.2f}")

    while stats[split]["kept"] < need_total:
        if disk_free_gb() < RESERVE_GB:
            print(f"[{split}] parando: disco livre abaixo da reserva ({RESERVE_GB} GB).")
            break

        if tile_idx >= len(df):
            # se acabou tiles, reembaralha e tenta de novo
            tile_idx = 0
            df = df.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000))).reset_index(drop=True)

        r = df.iloc[tile_idx]
        tile_idx += 1

        mosaic_path = str(r["mosaic_path"])
        gt_path = str(r["gt_path"])

        # quantos ainda faltam
        remaining = need_total - stats[split]["kept"]
        remaining_pos = need_pos - stats[split]["pos"]
        remaining_neg = need_neg - stats[split]["neg"]

        # lote pequeno por tile (pra não travar)
        want_pos_first = remaining_pos > 0
        batch = min(SHARD_SIZE, remaining, 64)

        X_list, Y_list = [], []

        # coleta positivos
        if remaining_pos > 0:
            n_pos_here = min(batch, remaining_pos, SHARD_SIZE)
            Xs, Ys = sample_from_one_tile(mosaic_path, gt_path, n_need=n_pos_here, want_pos=True)
            X_list.extend(Xs); Y_list.extend(Ys)

        # completa com negativos
        if len(X_list) < SHARD_SIZE and remaining_neg > 0:
            n_neg_here = min(SHARD_SIZE - len(X_list), remaining_neg)
            Xs, Ys = sample_from_one_tile(mosaic_path, gt_path, n_need=n_neg_here, want_pos=False)
            X_list.extend(Xs); Y_list.extend(Ys)

        if len(X_list) == 0:
            continue

        # salva shard único (X e Y juntos)
        X_arr = np.stack(X_list, axis=0)  # (N,H,W,C) uint16
        Y_arr = np.stack(Y_list, axis=0)  # (N,H,W) uint8

        out_file = OUT_ROOT / f"{split}_shard_{shard_idx:04d}.npz"
        np.savez_compressed(out_file, X=X_arr, Y=Y_arr)

        shard_idx += 1
        stats[split]["shards"] += 1
        stats[split]["kept"] += int(X_arr.shape[0])

        # atualiza pos/neg conforme o critério usado (pos = patch com >= MIN_POS_PIXELS)
        pos_mask = (Y_arr == 1).sum(axis=(1,2)) >= MIN_POS_PIXELS
        stats[split]["pos"] += int(pos_mask.sum())
        stats[split]["neg"] += int((~pos_mask).sum())

        if stats[split]["kept"] % 200 == 0 or stats[split]["kept"] >= need_total:
            print(f"[{split}] kept={stats[split]['kept']}/{need_total} pos={stats[split]['pos']} neg={stats[split]['neg']} free_gb={disk_free_gb():.2f}")

print("\nDONE. saída em:", OUT_ROOT)
print("stats:", stats)
print("free_gb final:", disk_free_gb())

removed: /content/unet_patches
split_norm counts:
 split_norm
train    118
test      26
val       25
Name: count, dtype: int64
approx MB/patch @ 256: 2.69 MB
free_gb=23.03 -> max_patches~8201 (com reserve 1.5 GB)
TARGET final: {'train': 3000, 'val': 600, 'test': 600}

[train] target=3000 (pos=1500 neg=1500) tiles=118 free_gb=23.03


/tmp/ipython-input-4264335547.py:56: RuntimeWarning: overflow encountered in scalar subtract
  return int(A - B - C + D)
/tmp/ipython-input-4264335547.py:56: RuntimeWarning: overflow encountered in scalar add
  return int(A - B - C + D)


[train] kept=800/3000 pos=160 neg=640 free_gb=22.22
[train] kept=1600/3000 pos=288 neg=1312 free_gb=22.31
[train] kept=3000/3000 pos=1500 neg=1500 free_gb=19.58

[val] target=600 (pos=300 neg=300) tiles=25 free_gb=19.58
[val] kept=600/600 pos=300 neg=300 free_gb=19.45

[test] target=600 (pos=300 neg=300) tiles=26 free_gb=19.45
[test] kept=600/600 pos=300 neg=300 free_gb=18.65

DONE. saída em: /content/unet_patches_v4
stats: {'train': {'kept': 3000, 'pos': 1500, 'neg': 1500, 'shards': 99}, 'val': {'kept': 600, 'pos': 300, 'neg': 300, 'shards': 20}, 'test': {'kept': 600, 'pos': 300, 'neg': 300, 'shards': 20}}
free_gb final: 18.65264892578125
